We will talk frequently about stochastic processes on this blog. They are fundamental to quantitative finance, and a basic understanding of stochastic calculus will be an important prerequisite for many of the topics considered later.

We start with discrete-time stochastic processes, in which the time variable takes values in the non-negative integers. In a later post, we will turn to continuous-time processes, where time takes values in the non-negative real numbers, as in the case of Brownian motion.

We begin by introducing the fundamental object of probability theory: the probability space, denoted by

$ (\Omega,\mathcal F,\mathbb P). $

This three-tuple provides a complete mathematical description of randomness, consisting of a non-empty set $\Omega$ that defines a sample space (i.e., the set of all possible outcomes of the experiment), a $\sigma$-algebra $\mathcal{F}$ of subsets of $\Omega$, which defines the event space (i.e., the collection of all events to which a probability can be assigned), and a probability measure $\mathbb{P}$ on $(\Omega, \mathcal{F})$, which assigns a probability to each event in $\mathcal{F}$.

To build intuition, consider the example of rolling a fair die.. For a single roll of a fair die, we have

$ \Omega = {1,2,3,4,5,6}, \qquad \mathcal F = 2^\Omega, \qquad \mathbb P({\omega}) = \frac{1}{6} \quad \text{for each } \omega \in \Omega. $

Here, $2^\Omega$ denotes the set of all subsets of $\Omega$, meaning every possible event is measurable. To move toward stochastic processes, we consider repeated experiments. For example, if we roll a die infinitely many times, the sample space becomes

$ \Omega = {1,2,3,4,5,6}^{\mathbb N}. $

An element $ \omega \in \Omega $ is now an infinite sequence

$ \omega = (\omega_1, \omega_2, \omega_3, \dots), $

where $\omega_t $ represents the outcome of the die at time $ t $. In this setting, a single $ \omega $ describes an entire possible history of the experiment.

This perspective, where one outcome encodes a full evolution over time, is essential for understanding stochastic processes.

So what exactly is a stochastic process? In order to define a stochastic process, we need to call upon another concept from probability theory: the concept of a random variable. A random variable is simply a measurable function

$ X : (\Omega,\mathcal F) \longrightarrow (\mathbb R^d,\mathcal B(\mathbb R^d)), $

where $ \mathcal B(\mathbb R^d) $ is the Borel $\sigma$-algebra on $ \mathbb R^d $, consisting of all sets that can be constructed from open sets using countable unions, intersections, and complements.

Intuitively, a random variable takes an outcome $\omega \in \Omega$ and assigns it a numerical value in $ \mathbb R^d $. The measurability condition ensures that probabilities of events involving $ X $ are well-defined.

From the notion of a random variable, we can define a stochastic process as a family of random variables indexed by some set $T$ such that

$ X = {X_t : t \in T}, $

where each

$ X_t : (\Omega,\mathcal F) \longrightarrow (\mathbb R^d,\mathcal B(\mathbb R^d)) $

is a random variable.

In this post, we will restrict our attention away from interesting mathematical nuances (those can be discussed another time) and primarily consider continuous-time processes, where the index set is

$ T = \mathbb R_+ = [0,\infty). $

It is important to distinguish terminology: a process indexed by continuous time is called a continuous-time stochastic process. This does not imply that its paths are continuous. This because very intuitive and obvious upon pursuing a first-principle derivations of these concepts (which I recommend people explore; for those interested, consider [ref textbook]). For our present purposes, continuity of paths is a separate property that must be defined later.

A stochastic process can also be viewed as a function of two variables:

$ X : T \times \Omega \longrightarrow \mathbb R^d,
\qquad
(t,\omega) \longmapsto X_t(\omega).
$

This formulation highlights two complementary perspectives:

Fixing time $ t $ such that $ (X_t) $ is a random variable, describes the distribution of the process at time $ t $;
Fixing an outcome $ \omega $ such that the mapping $ t \longmapsto X_t(\omega) $ is known as a sample path, representing one possible evolution of the process over time.

Understanding this dual viewpoint, namely randomness across outcomes and evolution across time, is central to developing intuition for stochastic processes.

## Example
Here is an example of a stochastic process constructed from repeated independent die rolls, whose marginal distributions become approximately Gaussian as time increases.

In [2]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go

from plotly.subplots import make_subplots

# Mathematical constants

DIE_MEAN = 3.5
DIE_STANDARD_DEVIATION = np.sqrt(35.0 / 12.0)

# Simulation parameters

SEED = 42
N_STEPS = 100
N_DISPLAY_PATHS = 40
N_DISTRIBUTION_PATHS = 1_000

N_BINS = 30
FRAME_DURATION_MS = 80

# Functions

def symmetric_limits(
    values: np.ndarray,
    padding: float = 0.10,
) -> tuple[float, float]:
    """Return symmetric axis limits containing all supplied values."""
    values = np.asarray(values, dtype=float)

    if values.size == 0:
        raise ValueError("values must not be empty.")

    bound = float(np.max(np.abs(values)))

    if bound == 0.0:
        bound = 1.0

    padded_bound = (1.0 + padding) * bound
    return -padded_bound, padded_bound


def normal_density(
    values: np.ndarray,
    mean: float,
    variance: float,
) -> np.ndarray:
    """Evaluate the density of a normal distribution."""
    if variance <= 0.0:
        return np.zeros_like(values, dtype=float)

    return (
        np.exp(
            -0.5 * (values - mean) ** 2 / variance
        )
        / np.sqrt(2.0 * np.pi * variance)
    )


def simulate_die_walks(
    n_paths: int,
    n_steps: int,
    seed: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Simulate centred, variance-normalised die-driven random walks."""
    rng = np.random.default_rng(seed)

    rolls = rng.integers(
        low=1,
        high=7,
        size=(n_paths, n_steps),
    )

    standardised_rolls = (
        rolls - DIE_MEAN
    ) / DIE_STANDARD_DEVIATION

    increments = standardised_rolls / np.sqrt(n_steps)

    paths = np.zeros((n_paths, n_steps + 1))
    paths[:, 1:] = np.cumsum(increments, axis=1)

    times = np.linspace(0.0, 1.0, n_steps + 1)

    return times, paths

# Simulate one common ensemble

times, paths = simulate_die_walks(
    n_paths=N_DISTRIBUTION_PATHS,
    n_steps=N_STEPS,
    seed=SEED,
)

selected_path = paths[0]

path_limits = symmetric_limits(
    paths[:N_DISPLAY_PATHS],
)

distribution_limits = symmetric_limits(paths)

bin_edges = np.linspace(
    distribution_limits[0],
    distribution_limits[1],
    N_BINS + 1,
)

bin_centres = 0.5 * (
    bin_edges[:-1] + bin_edges[1:]
)

bin_width = bin_edges[1] - bin_edges[0]

density_grid = np.linspace(
    distribution_limits[0],
    distribution_limits[1],
    300,
)

# Empirical histograms and Gaussian approximations

frame_indices = np.arange(1, N_STEPS + 1)

empirical_densities = np.array(
    [
        np.histogram(
            paths[:, frame],
            bins=bin_edges,
            density=True,
        )[0]
        for frame in frame_indices
    ]
)

gaussian_densities = np.array(
    [
        normal_density(
            density_grid,
            mean=0.0,
            variance=times[frame],
        )
        for frame in frame_indices
    ]
)

maximum_density = max(
    float(empirical_densities.max()),
    float(gaussian_densities.max()),
)

density_axis_limit = 1.10 * maximum_density

# Initial frame

initial_frame = 1
initial_index = 0

initial_values = paths[:, initial_frame]
initial_mean = float(initial_values.mean())

# Construct figure

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.58, 0.42],
    horizontal_spacing=0.10,
    subplot_titles=(
        "Sample paths through time",
        "Distribution at a fixed time",
    ),
)


# Repeated realisations
for path_index in range(1, N_DISPLAY_PATHS):
    fig.add_trace(
        go.Scatter(
            x=times,
            y=paths[path_index],
            mode="lines",
            line={
                "width": 1,
            },
            opacity=0.18,
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=1,
    )


# Selected realised path
selected_path_trace_index = len(fig.data)

fig.add_trace(
    go.Scatter(
        x=times[: initial_frame + 1],
        y=selected_path[: initial_frame + 1],
        mode="lines",
        line={
            "width": 3,
        },
        name="Selected realisation",
    ),
    row=1,
    col=1,
)

# Current point on selected path
current_point_trace_index = len(fig.data)

fig.add_trace(
    go.Scatter(
        x=[times[initial_frame]],
        y=[selected_path[initial_frame]],
        mode="markers",
        marker={
            "size": 10,
        },
        name="Current value",
    ),
    row=1,
    col=1,
)


# Empirical histogram
histogram_trace_index = len(fig.data)

fig.add_trace(
    go.Bar(
        x=empirical_densities[initial_index],
        y=bin_centres,
        width=bin_width,
        orientation="h",
        opacity=0.65,
        name="Empirical distribution",
    ),
    row=1,
    col=2,
)


# Gaussian approximation
gaussian_trace_index = len(fig.data)

fig.add_trace(
    go.Scatter(
        x=gaussian_densities[initial_index],
        y=density_grid,
        mode="lines",
        line={
            "width": 3,
        },
        name="Gaussian approximation",
    ),
    row=1,
    col=2,
)

# Empirical mean
mean_trace_index = len(fig.data)

fig.add_trace(
    go.Scatter(
        x=[0.0, density_axis_limit],
        y=[initial_mean, initial_mean],
        mode="lines",
        line={
            "width": 2,
            "dash": "dash",
        },
        name="Empirical mean",
    ),
    row=1,
    col=2,
)

# Selected value on distribution panel
selected_density = normal_density(
    np.array([selected_path[initial_frame]]),
    mean=0.0,
    variance=times[initial_frame],
)[0]

distribution_point_trace_index = len(fig.data)

fig.add_trace(
    go.Scatter(
        x=[selected_density],
        y=[selected_path[initial_frame]],
        mode="markers",
        marker={
            "size": 10,
        },
        name="Selected path value",
    ),
    row=1,
    col=2,
)

# Animation frames

frames = []

animated_trace_indices = [
    selected_path_trace_index,
    current_point_trace_index,
    histogram_trace_index,
    gaussian_trace_index,
    mean_trace_index,
    distribution_point_trace_index,
]

for density_index, frame in enumerate(frame_indices):
    current_time = times[frame]
    current_selected_value = selected_path[frame]
    current_values = paths[:, frame]
    current_mean = float(current_values.mean())

    selected_value_density = normal_density(
        np.array([current_selected_value]),
        mean=0.0,
        variance=current_time,
    )[0]

    frames.append(
        go.Frame(
            name=str(frame),
            traces=animated_trace_indices,
            data=[
                go.Scatter(
                    x=times[: frame + 1],
                    y=selected_path[: frame + 1],
                ),
                go.Scatter(
                    x=[current_time],
                    y=[current_selected_value],
                ),
                go.Bar(
                    x=empirical_densities[density_index],
                    y=bin_centres,
                    width=bin_width,
                    orientation="h",
                ),
                go.Scatter(
                    x=gaussian_densities[density_index],
                    y=density_grid,
                ),
                go.Scatter(
                    x=[0.0, density_axis_limit],
                    y=[current_mean, current_mean],
                ),
                go.Scatter(
                    x=[selected_value_density],
                    y=[current_selected_value],
                ),
            ],
            layout=go.Layout(
                title_text=(
                    "Die-driven stochastic process"
                    
                )
            ),
        )
    )

fig.frames = frames

# Layout and controls

fig.update_layout(
    title={
        "text": (
            "Die-driven stochastic process"
        ),
        "x": 0.5,
        "y": 0.98,
        "xanchor": "center",
        "yanchor": "top",
    },
    width=1100,
    height=550,
    template="plotly_white",
    bargap=0.02,
    hovermode="closest",
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.20,
        "xanchor": "center",
        "x": 0.5,
    },
    margin={
        "l": 70,
        "r": 40,
        "t": 145,
        "b": 80,
    },
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "x": 0.5,
            "y": -0.14,
            "xanchor": "center",
            "yanchor": "top",
            "showactive": False,
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": FRAME_DURATION_MS,
                                "redraw": True,
                            },
                            "transition": {
                                "duration": 0,
                            },
                            "fromcurrent": True,
                            "mode": "immediate",
                        },
                    ],
                },
                {
                    "label": "❚❚ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {
                                "duration": 0,
                                "redraw": False,
                            },
                            "transition": {
                                "duration": 0,
                            },
                            "mode": "immediate",
                        },
                    ],
                },
            ],
        }
    ],
)


# Left panel
fig.update_xaxes(
    title_text=r"Normalised time (t)",
    range=[times[0], times[-1]],
    showgrid=True,
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text=r"Process value X_t",
    range=list(path_limits),
    showgrid=True,
    row=1,
    col=1,
)


# Right panel
fig.update_xaxes(
    title_text="Probability density",
    range=[0.0, density_axis_limit],
    showgrid=True,
    row=1,
    col=2,
)

fig.update_yaxes(
    title_text=r"Value of X_t",
    range=list(distribution_limits),
    showgrid=True,
    row=1,
    col=2,
)


fig.show()

# Create html for blog

fig.write_html(
    "die_driven_stochastic_process.html",
    include_plotlyjs="cdn",
    full_html=True,
    config={
        "displayModeBar": False,
        "responsive": True,
    },
)

The selected path represents one complete simulated trajectory of the stochastic process, corresponding to a single outcome $ \omega $, while the current value is the point $ X_{t_k}(\omega) $ on that path at the time displayed by the animation. We place the $ N $ discrete steps on the unit time interval by defining $t_k = k/N $. Thus, $t_k$ is a dimensionless, normalised time variable representing the proportion of the simulation horizon that has elapsed.
In the left-hand panel, the line shows the selected path as it evolves through time and the marker identifies its current position. In the right-hand panel, the same current value is shown relative to the empirical distribution generated from all simulated paths and the accompanying Gaussian approximation, allowing us to compare one realised outcome with the distribution of possible outcomes at that fixed time.


## Properties of stochastic processes

Having introduced stochastic processes in the previous section, we now focus on their key properties. These properties describe how a process behaves over time, how its values are distributed, and how different time points are related.

The main properties of a stochastic process describe either:

$ \text{the behaviour of } t \mapsto X_t(\omega) $

for a fixed outcome $\omega$, or

$ \text{the joint distribution of } X_{t_1}, \ldots, X_{t_n} $

at several fixed times.

### Discrete-time and continuous-time processes

The index set $T$ determines whether the process is discrete-time or continuous-time.

As described in the name, a discrete-time stochastic process is indexed by a countable set of times, usually

$ T = \mathbb N_0 = \{0,1,2,\ldots \}. $

For example, if $D_1, D_2, \ldots$ are successive die rolls and

$ \xi_k = \frac{D_k - \frac{7}{2}}{\sqrt{35/12}}, $

then the partial sums

$ S_n = \sum_{k=1}^{n} \xi_k, \qquad S_0 = 0, $

form a discrete-time stochastic process.

A continuous-time stochastic process, on the other hand, is indexed by an interval of real numbers, usually

$ T = \mathbb R_+ = [0,\infty) $

or a finite interval such as

$ T = [0, T_0]. $

The term continuous-time refers only to the index set. It does not imply that the sample paths are continuous.

In the simulation, the scaled random walk is defined on the grid

$ t_k = \frac{k}{N}, \qquad k = 0,1,\ldots,N, $

so it is a discrete-time process defined on a finite grid contained in $ [0,1] $.

### Discrete-state and continuous-state processes

The state space describes the possible values taken by the process. A process is discrete-state if its values lie in a finite or countable set. For example,

$ S_n = \varepsilon_1 + \cdots + \varepsilon_n, \qquad \varepsilon_k \in \{-1, +1 \}, $
takes values in $\mathbb Z $.

A process is continuous-state if its values lie in an uncountable set such as $\mathbb R$ or $\mathbb R^d$.

Thus, a process is classified by both its time index and its state space.

### Sample paths

For a fixed outcome $ \omega \in \Omega $, the function

$ t \mapsto X_t(\omega) $

is called a sample path or trajectory. As we have seen, a stochastic process represents all possible sample paths. A single plotted curve corresponds to just one independent path realisation.

In the die-roll example, moreover, a particular sequence

$ \omega = (D_1, D_2, \ldots, D_N) $

can be seen to determine a path such that

$ X_{t_k}(\omega) = \frac{1}{\sqrt N} \sum_{j=1}^{k} \xi_j(\omega), $

with dfferent outcomes clearly producing different paths.

### Finite-dimensional distributions

A stochastic process can be described through its values at finitely many times. For times $ t_1 < t_2 < \cdots < t_m, $ the random vector $
(X_{t_1}, \ldots, X_{t_m}) $ has a joint distribution on $ (\mathbb R^d)^m $. The collection of all such joint distributions is called the collection of finite-dimensional distributions of the process. For a single time $ t $, $ \mathcal L(X_t) $ is the marginal distribution.

In the simulation, the histogram at time $t $ approximates $ \mathcal L(X_t) $. As $ t_k $ increases, more independent increments are accumulated. The distribution spreads out and, when the number of accumulated increments is sufficiently large, becomes approximately Gaussian. 

A process is a Gaussian process if every finite collection of its values has a multivariate normal distribution. In the example of the die-roll process, it is not exactly Gaussian since it is discrete, but its marginal distributions become approximately Gaussian as the number of increments increases. Approximately Gaussian marginal distributions at individual times do not, by themselves, establish that the process is approximately Gaussian in the stronger finite-dimensional sense.

### Mean function

If $ X_t $ is integrable, the mean function is

$ m_X(t) = \mathbb E[X_t], $

where on the right-hand side of the equality we have the expectation of the distribution.

For the scaled die-roll process, since $ \mathbb E[\xi_k] = 0 $ it follows $
\mathbb E[X_{t_k}] = 0. $ Thus, the mean function is zero, even though individual paths may deviate significantly from zero.

### Variance and covariance functions

The variance at time $ t $ is

$ \text{Var}(X_t) = \mathbb E[(X_t - \mathbb E[X_t])^2]. $

The covariance function is

$ \text{Cov}(X_s, X_t) = \mathbb E[(X_s - \mathbb E[X_s])(X_t - \mathbb E[X_t])]. $

For the scaled die-roll process,

$ \text {Var}(X_{t_k}) = \frac{k}{N} = t_k, $

and

$ \text{Cov}(X_{t_k}, X_{t_\ell}) = \min(t_k, t_\ell), $

which holds for $ t_k = k/N $ and $t_l = l/N $.

This reflects the fact that values at different times share common increments. For $ s < t $, the difference $ X_t - X_s $ is called an increment. For the scaled die-roll process, increments are sums of independent variables corresponding to new die rolls.

A process has independent increments if increments over disjoint intervals are independent. The scaled die-roll process has independent increments because each increment depends on a distinct set of independent die rolls.

A process has stationary increments if the distribution of $ X_{t+h} - X_t $ depends only on $ h $, not on $ t $.  For the die roll process, only grid compatible increments are defined since its increments depend only on the number $ h $ of steps.

This leads us to different notions of stationarity. For instance, a process is strictly stationary if all finite-dimensional distributions are invariant under time shifts. For the scaled die-roll process, it is not strictly stationary because its distribution changes over time.

A process is weakly stationary if:

* $ \mathbb E[X_t] $ is constant,
* $ \text{Cov}(X_s, X_t) $ depends only on $ t - s $.

In the case of the scaled die-roll process, it is clearly not weakly stationary because its variance depends on $ t $.

## Continuity of sample paths

Importantly, a process has continuous sample paths if $ t \mapsto X_t(\omega) $ is continuous for almost every $ \omega $. In our example of the scaled die roll process, since it is defined only on a finite discrete grid, continuity in probability is not directly applicable unless a continuous-time interpolation or extension is introduced.

Furthermore, it is notable that a generic process is continuous in probability at $ t $ if

$ \lim_{s \to t} \mathbb P(|X_s - X_t| > \varepsilon) = 0 $

for every $ \varepsilon > 0 $. This is a weaker statement than pathwise continuity.

## Markov property

A process has the Markov property if the future is conditional on the present state, such that the conditional distribution of the future does not depend on the earlier history. For the scaled die-roll process,

$ X_{t_{k+1}} = X_{t_k} + \frac{\xi_{k+1}}{\sqrt N}, $

and $ \xi_{k+1} $ is independent of the past, so the process is technically Markov. We'll discuss the theory of Markov processes in extensive detail in future entries.

## 5. Filtrations and information

Finally, we arrive at one other important concept: the idea that, in order to study how information evolves over time, we requires filtrations.

A filtration is a family of $ \sigma $-algebras $ (\mathcal F_t)_{t \in T} $ such that $ \mathcal F_s \subseteq \mathcal F_t \quad \text{whenever } s \leq t. $ Each $ \mathcal F_t $ represents the information available up to time $ t $.

For the die roll filteration $ \mathcal{F}_0 = \{ \emptyset, \Omega \} $, we have $ \mathcal F_k = \sigma(D_1, \ldots, D_k) $ for $ k \geq 1 $. This forms a filtration representing the accumulated information.

A process is adapted if $ X_t $ is measurable with respect to $ \mathcal F_t $. Adaptedness permits dependence on present information as well as past information.Hence, the scaled die roll process is an adapted stochastic process because $ X_{t_k} $ depends only on the die rolls observed up to and including time $ t $ (i.e., a discrete-time filtration). This means that for any given roll, all preceding outcomes are publicly observable, and you can only make decisions for future rolls based on the history of previous outcomes.

### The natural filtration

The natural filtration of a process is

$ \mathcal F_t^X = \sigma(X_s : s \leq t), $

which for the discrete grid become $ \mathcal{F}_{t_k}^X = \sigma (X_{t_0} \dots X_{t_k}) $. 

In general, a natural filtration represents all information revealed by the process itself.

At time $ t $, $ \mathcal F_t $ contains past and present information, but not future events. Conditional expectations such as $\mathbb E[X_t \mid \mathcal F_s] $ where $ s \leq t $, describes predictions based on available information.

In finance, $ \mathcal F_t $ often represents market information. Models require that decisions depend only on current information (i.e., non-anticipation).

## Filtrations, the Markov property, and martingales

A few last comments. In the context of filtrations, the Markov property can be expressed as

$ \mathbb P(X_{t+h} \in A \mid \mathcal F_t) = \mathbb P(X_{t+h} \in A \mid X_t) $

in the context of the natural filtration or a filtration relative to which the process is Markov. 

Filtrations also allow us to define martingales, which satisfy

$ \mathbb E[M_t \mid \mathcal F_s] = M_s, \ s \leq t. $

A martingale is a stochastic process where the expected value of the next observation, based on all previous observations, is equal to the current value. Mathematically, a martingale requires the following properties be satisfied:

* $ M_t $ is $ \mathcal{F}_t $-measurable;
* $ \mathbb{E}[\mid M_t \mid] < \infty $;

Martingales are used, for example, to model a "fair game" with no predictable upward or downward drift, forming the foundation of risk-neutral pricing in finance.

# Conclusions

We have introduced key properties of stochastic processes and the concept of filtrations, which describe the flow of information. The central organising idea was based on two perspectives: fix $ \omega = \text{study a sample path} $ and fix $ t = \text{study the distribution of} X_t$.

The next topics will develop tools such as moment-generating functions, characteristic functions, and conditional expectation, leading to martingales and ultimately to Brownian motion.
